In [ ]:
import pandas as pd
import numpy as np
import os

# --- KULLANICI AYARLARI ---
target_gene = 'CTNNB1'  # 'MYC' için burayı değiştirmeyi unutma

base_path = '../data/03_fingerprints/'
save_path = '../data/04_raw_bit_matrices/'

# Klasör kontrolü
os.makedirs(save_path, exist_ok=True)

# Sütun tiplerini sabitleme (Overflow hatası almamak için)
dtype_dict = {
    'PUBCHEM_SID': str, 
    'PUBCHEM_CID': str, 
    'LABEL': int, 
    'Morgan_Fingerprint_2048': str
}

# 1. Verileri oku
print(f"[>] Loading raw data for {target_gene}...")
df_active = pd.read_csv(os.path.join(base_path, f'{target_gene}_active.csv'), 
                        usecols=['PUBCHEM_SID', 'PUBCHEM_CID', 'LABEL', 'Morgan_Fingerprint_2048'], 
                        dtype=dtype_dict)
df_inactive = pd.read_csv(os.path.join(base_path, f'{target_gene}_inactive.csv'), 
                          usecols=['PUBCHEM_SID', 'PUBCHEM_CID', 'LABEL', 'Morgan_Fingerprint_2048'], 
                          dtype=dtype_dict)

# 2. Birleştir
df = pd.concat([df_active, df_inactive], ignore_index=True)

# 3. Yinelenenleri kaldır (Deduplication)
initial_count = len(df)
df = df.drop_duplicates(subset=['PUBCHEM_CID'])
final_count = len(df)

# --- Summary Çıktısı ---
print(f"\n--- Summary ---")
print(f"Total Rows: {initial_count}")
print(f"After Deduplication: {final_count}")
print(f"Removed Duplicates: {initial_count - final_count}")

df.head()

[>] Loading raw data for CTNNB1...

--- Summary ---
Total Rows: 1100
After Deduplication: 715
Removed Duplicates: 385


,PUBCHEM_SID,PUBCHEM_CID,LABEL,Fingerprint
0,471061810.0,164616842.0,1,0000000000000010000010000000000000000000000000...
1,471061811.0,164627981.0,1,0000000000000010000010000000000000000000000000...
2,471061812.0,164617144.0,1,0000000000000010000010000000000000000000000000...
3,471061816.0,164625345.0,1,0000000000000010000010000000000000000000000000...
4,471061817.0,164623731.0,1,0000000000000010000010000000000000000000000000...


**Bit Splitting and Save (ECFP4 - 2048 bit)**

In [ ]:
# 1. Bit Splitting (ECFP4 - 2048 bit)
print(f"[>] {target_gene} için bit splitting (ECFP4 - 2048 bit) başlatıldı...")
X_bits = pd.DataFrame(df['Morgan_Fingerprint_2048'].apply(list).tolist()).astype(int)
X_bits.columns = [str(i) for i in range(2048)]
X_bits['class'] = df['LABEL'].values
X_bits = X_bits.copy() 

# 2. Kaydet (ECFP4 ibaresi eklendi)
orig_file = os.path.join(save_path, f"{target_gene}_ecfp4_2048_original.csv")
X_bits.to_csv(orig_file, index=False)
print(f"[✓] ECFP4 original binary data saved: {orig_file}")

[>] CTNNB1 için bit splitting (ECFP4 - 2048 bit) başlatıldı...


/tmp/ipykernel_1418130/4123807730.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_bits['class'] = df['LABEL'].values


[✓] ECFP4 original binary data saved: /home/beyza/tubitak_beyza/pubchem/data_set_final/CTNNB1_ecfp4_2048_original.csv


**Correlation Removal and Save**

Ön analizlerde veri sızıntısı riski fark edilmiş ve kod mimarisi sızıntısız (leakage-free) olacak şekilde refactor edilmiştir; ancak mevcut biyoinformatik bulguların sürekliliği adına üretim modeli sabit tutulmuştur.

In [ ]:
# def drop_highly_correlated_features(df, threshold=0.8):
#     print(f"[>] {threshold} eşik değeri ile korelasyon matrisi hesaplanıyor...")
#     # Sadece bit sütunlarını alıyoruz (class hariç)
#     correlation_matrix = df.iloc[:, :-1].corr().abs()
    
#     # Üst üçgeni maskeleme
#     upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    
#     # Threshold üzeri olan sütunları bul
#     to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > threshold)]
    
#     print(f"[✓] {len(to_drop)} adet yüksek korelasyonlu sütun tespit edildi.")
#     return df.drop(columns=to_drop)


# # İşlemi uygula (Threshold 0.8 hoca standardı)
# X_clean = drop_highly_correlated_features(X_bits, threshold=0.8)

# # Final dosyasını kaydet
# clean_file = os.path.join(save_path, f"{target_gene}_ecfp4_2048_no_corr.csv")
# X_clean.to_csv(clean_file, index=False)

# print(f"\n--- Final Summary: {target_gene} ---")
# print(f"Descriptor: ECFP4 (Morgan Fingerprint)")
# print(f"Original Bits: 2048")
# print(f"Cleaned Bits: {X_clean.shape[1]-1}")
# print(f"Saved to: {clean_file}")

[>] 0.8 eşik değeri ile korelasyon matrisi hesaplanıyor...


[✓] 577 adet yüksek korelasyonlu sütun tespit edildi.

--- Final Summary: CTNNB1 ---
Descriptor: ECFP4 (Morgan Fingerprint)
Original Bits: 2048
Cleaned Bits: 1471
Saved to: /home/beyza/tubitak_beyza/pubchem/data_set_final/CTNNB1_ecfp4_2048_no_corr.csv
